#### Download The Data

In [1]:
import requests

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

response = requests.get(url)
response.raise_for_status()

In [3]:
with open("../data/input.txt", 'w', encoding="utf-8") as f:
    f.write(response.text)

print("Dataset Downloaded Successfully.")

Dataset Downloaded Successfully.


#### Dataset Validation

In [4]:
from pathlib import Path

DATA_PATH = Path("../data/input.txt")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not Found: {DATA_PATH}")

text = DATA_PATH.read_text(encoding='utf-8')

print(f"Dataset Loaded Successfully.")
print(f"Number of Characters: {len(text):,}")
print(f"\nFirst 500 Characters: \n\n{text[:500]}")

Dataset Loaded Successfully.
Number of Characters: 1,115,394

First 500 Characters: 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [5]:
print("Empty Dataset:", len(text) == 0)

Empty Dataset: False


In [6]:
print("Contains newline:", "\n" in text)

Contains newline: True


In [7]:
print("Contains Spaces:", " " in text)

Contains Spaces: True


In [8]:
print("Contains alphabetic characters:", any(c.isalpha() for c in text))

Contains alphabetic characters: True


#### Build The Vocabulary

In [9]:
chars = sorted(set(text))

vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(chars)

Vocabulary size: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


We are using character level tokenization for this project, but usually large language models use subword tokenization like BPE.


#### Create the tokenizer mappings

In [10]:
# String to Integer:

stoi = {ch: i for i,ch in enumerate(chars)}

In [11]:
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [12]:
# Integer to String:

itos = {i: ch for i, ch in enumerate(chars)}

In [13]:
print(itos)

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


The two mappings are inverses to each other

#### Build Encode and Decode

In [14]:
def encode(text):
    return [stoi[ch] for ch in text] # We are creating the list here

def decode(ids):
    return "".join(itos[i] for i in ids)

In [15]:
# Test

sample_text = "Hello World"

encoded = encode(sample_text)
decoded = decode(encoded)

In [16]:
print("Original:", sample_text)
print("Encoded:", encoded)
print("Decoded:", decoded)

Original: Hello World
Encoded: [20, 43, 50, 50, 53, 1, 35, 53, 56, 50, 42]
Decoded: Hello World


In [17]:
# Validate the decoder

assert decode(encode(sample_text)) == sample_text

print("Tokenizer round-trip test passed.")

Tokenizer round-trip test passed.


#### Language Modelling

We don't want to randomly shuffle individual characters and then split them because it will destroy the sequential structure and can introduce leakage.

So we will split the continuous text

In [18]:
split_idx = int(0.9 * len(text))

train_text = text[:split_idx]
val_text = text[split_idx:]

In [19]:
print(f"Training Characters: {len(train_text):,}")
print(f"Validation Characters: {len(val_text):,}")

Training Characters: 1,003,854
Validation Characters: 111,540


In [21]:
# Encode the datasets
import torch

train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids = torch.tensor(encode(val_text), dtype=torch.long)

d:\mini-gpt\.mini-gpt-venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [22]:
print(f"Train: {train_ids.shape}")
print(f"Validation: {val_ids.shape}")

Train: torch.Size([1003854])
Validation: torch.Size([111540])


These are not training samples yet. It is one big sequence of character Ids.

In [23]:
# Define the Context Length or Block size
# Context length means how many previous characters should the model Look at when predicting the next character.

block_size = 64

In [24]:
# Test

x = train_ids[:block_size]
y = train_ids[1:block_size + 1]

print("x: ", x)
print("y: ", y)

x:  tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50])
y:  tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,  1,
        44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1, 57,
        54, 43, 39, 49,  8,  0,  0, 13, 50, 50])


In [25]:
def get_batch(data, batch_size, block_size):

    max_start = len(data) - block_size
    
    ix = torch.randint(
        0,
        max_start,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size] for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1] for i in ix
    ])

    return x,y

In [26]:
batch_size = 4
block_size = 64

x, y = get_batch(
    train_ids,
    batch_size,
    block_size
)

print("X Shape: ", x.shape)
print("Y Shape: ", y.shape)

X Shape:  torch.Size([4, 64])
Y Shape:  torch.Size([4, 64])


In [27]:
print(x)

tensor([[46, 39, 57, 58, 43, 57,  1, 53, 59, 56,  1, 51, 39, 56, 56, 47, 39, 45,
         43,  6,  0, 32, 53,  1, 57, 58, 53, 54,  1, 58, 46, 43,  1, 47, 52, 59,
         52, 42, 39, 58, 47, 53, 52,  1, 53, 44,  1, 46, 43, 56,  1, 58, 43, 39,
         56, 57, 11,  0, 35, 46, 47, 41, 46,  6],
        [ 0, 26, 53, 56,  1, 51, 53, 58, 46, 43, 56,  6,  1, 61, 47, 44, 43,  6,
          1, 52, 53, 56,  1, 17, 52, 45, 50, 39, 52, 42,  5, 57,  1, 41, 53, 59,
         52, 58, 43, 42,  1, 55, 59, 43, 43, 52,  8,  0,  0, 24, 27, 30, 16,  1,
         31, 32, 13, 26, 24, 17, 37, 10,  0, 18],
        [53, 42,  1, 42, 43, 43, 42,  1, 42, 63, 47, 52, 45,  1, 58, 53, 52, 45,
         59, 43, 50, 43, 57, 57,  0, 31, 50, 39, 59, 45, 46, 58, 43, 56, 57,  1,
         39,  1, 58, 46, 53, 59, 57, 39, 52, 42,  1, 61, 39, 47, 58, 47, 52, 45,
          1, 59, 54, 53, 52,  1, 58, 46, 39, 58],
        [43, 10,  1, 40, 59, 58,  1, 46, 43,  1, 56, 43, 58, 59, 56, 52, 57,  6,
          0, 31, 54, 50, 47, 58, 58, 47,

There are 4 sequences in the batch. Each Sequence has 64 characters/tokens. Each number the interger ID of a character from the vocabulary.

At this point we have completed:
1. Building a Vocalbulary
2. Assign an Integer ID to every character.
3. Encode the Text / Tokenization
4. Create Sequences of 64 token IDs.

In [28]:
x.shape

torch.Size([4, 64])

In [29]:
import torch.nn as nn

d_model = 128 # Dimensions of an Embedding vector

# Define the Embedding Matrix
token_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

In [30]:
token_embedding

Embedding(65, 128)

In [31]:
# Convert Our Ids into Embeddings

token_embeddings = token_embedding(x)

In [32]:
print("Token Ids Shape: ", x.shape)
print("Token Embeddings shape: ", token_embeddings.shape)

Token Ids Shape:  torch.Size([4, 64])
Token Embeddings shape:  torch.Size([4, 64, 128])


The same Token ID always maps to the same embedding vector at a given point in training.

#### Positional Embeddings

We do this because the self attention itself doesn't know that token 10 comes before token 20.

In [33]:
position_embedding = nn.Embedding(
    num_embeddings=block_size,
    embedding_dim=d_model
)

In [34]:
positions = torch.arange(block_size)

print(positions)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63])


In [35]:
pos_embeddings = position_embedding(positions)

In [36]:
pos_embeddings.shape

torch.Size([64, 128])

In [37]:
x_emb = token_embeddings + pos_embeddings

In [38]:
x_emb.shape

torch.Size([4, 64, 128])

We will give thim combined embedding matrix as input to the transformer.

#### Self-Attention:

At this point we have a Combined representation embedding matrix for the input tokens.

The main goal of self attention is to identify how much of relevance does each word has towards the current word.

Self Attention allows every position to look at other positions and determine **how much attention to give them**.

Now we will create three different representations for the input embeddings. These three representations are **QUERY, KEY and VALUE matrices**. Ideally we give weights to the input matrix and combined with weights we get the three matrices.

So basically Q,K and V are three different learned transformations of the same input representations.

In [39]:
# Create Q,K and V

import torch.nn.functional as F
import math

d_model = 128

q_proj = nn.Linear(d_model, d_model)
k_proj = nn.Linear(d_model, d_model)
v_proj = nn.Linear(d_model, d_model)

Q = q_proj(x_emb)
K = k_proj(x_emb)
V = v_proj(x_emb)

print(Q.shape)
print(K.shape)
print(V.shape)

torch.Size([4, 64, 128])
torch.Size([4, 64, 128])
torch.Size([4, 64, 128])


Now we calculate the attention scores

In [40]:
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_model)

print(scores.shape)

torch.Size([4, 64, 64])


Now we will apply the mask as we want **causal self attention** which means the model will have access to the current as well as the previous information only. Nothing about the future can be seen by the model.

In [41]:
# Create The mask

causal_mask = torch.tril(
    torch.ones(block_size, block_size) # It keeps the lower triangle intact and changes every other element to 0.
)

print(causal_mask)

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 1., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])


We do above masking because we want causal self attention. Suppose the sentence is `I Love Machine Learning.`

```text
        I  love  machine  learning
I       1   0      0        0
love    1   1      0        0
machine 1   1      1        0
learning1   1      1        1
```

In [42]:
# Now we will use this mask for our attention scores

scores = scores.masked_fill(
    causal_mask == 0,
    float("-inf")
)

`masked_fill()` replaces the attention scores at positions where the causal mask is 0 with -∞, so after softmax those future-token positions receive zero attention.

`-inf` because when we apply softmax, the model will give 0 attention to those values as the `softmax(-inf) = 0.00`.

Now the model gives Zero Attention to the Future positions.

In [43]:
print(scores.shape)

torch.Size([4, 64, 64])


In [44]:
# Now we apply softmax

attention_weights = F.softmax(scores, dim=-1)

In [45]:
print(attention_weights.shape)

torch.Size([4, 64, 64])


Now we will use these attention weights to get the attention values from the Value matrix.

Attention = AttentionWeights x V

In [46]:
attention_output = attention_weights @ V

print("Attention Weights: ", attention_weights.shape)
print("V: ", V.shape)
print("Attention_output: ", attention_output.shape)

Attention Weights:  torch.Size([4, 64, 64])
V:  torch.Size([4, 64, 128])
Attention_output:  torch.Size([4, 64, 128])


We return to our initial shape of d_model which is 128.

This was single head attention, now we will transform this into multi head attention so that the model can learn different types of relationships simultaneously.

#### Multi Head Attention.

In [47]:
num_heads = 4
head_dim = d_model // num_heads

B, T, C = Q.shape

Q = Q.view(B, T, num_heads, head_dim).transpose(1, 2)
K = K.view(B, T, num_heads, head_dim).transpose(1, 2)
V = V.view(B, T, num_heads, head_dim).transpose(1, 2)

print(Q.shape)
print(K.shape)
print(V.shape)

torch.Size([4, 4, 64, 32])
torch.Size([4, 4, 64, 32])
torch.Size([4, 4, 64, 32])


In [48]:
scores = Q @ K.transpose(-2, -1)

In [49]:
scores.shape

torch.Size([4, 4, 64, 64])

4 batches
×
4 heads
×
64 query positions
×
64 key positions

In [50]:
scores = scores / math.sqrt(head_dim)

In [52]:
scores = scores.masked_fill(
    causal_mask == 0,
    float("-inf")
)

In [53]:
attention_output = attention_weights @ V

In [54]:
attention_output = attention_output.transpose(1, 2)

In [55]:
attention_output = attention_output.contiguous().view(
    B, T, d_model
)

In [56]:
out_proj = nn.Linear(d_model, d_model)

attention_output = out_proj(attention_output)

In [57]:
attention_output.shape

torch.Size([4, 64, 128])

#### Convert the standalone attention code into actual trainable `CasualSelfAttention` module.

In [89]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x, return_attention=False):
        B, T, C = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)

        scores = scores / math.sqrt(self.head_dim)

        scores = scores.masked_fill(
            self.causal_mask[:T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(scores, dim=-1)

        attention_output = attention_weights @ V

        attention_output = attention_output.transpose(1, 2)

        attention_output = attention_output.contiguous().view(
            B, T, C
        )

        output = self.out_proj(attention_output)

        if return_attention:
            return output, attention_weights

        return output

In [90]:
# Instantiate it

attention = CausalSelfAttention(
    d_model = 128,
    num_heads = 4,
    block_size = 64
)

Now, we will build our Transformer Block

```text
x
 ↓
Multi-Head Self-Attention
 ↓
Residual Connection + LayerNorm
 ↓
Feed-Forward Network
 ↓
Residual Connection + LayerNorm
 ↓
output
```

In [141]:
# Model Configuration:

config = {
    "vocab_size": vocab_size,
    "d_model": 128,
    "num_heads": 8,
    "num_layers": 4,
    "block_size": 64,
    "ffn_dim": 512
}

In [150]:
# Implement the Feed-Forward Network

class FeedForward(nn.Module):

    def __init__(self, d_model, ffn_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [151]:
# Now we build the Transformer Block

class TransformerBlock(nn.Module):

    def __init__(self, d_model, num_heads, block_size, ffn_dim):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)

        self.attention = CausalSelfAttention(
            d_model,
            num_heads,
            block_size
        )

        self.ln2 = nn.LayerNorm(d_model)

        self.ffn = FeedForward(d_model, ffn_dim)

    def forward(self, x):

        # Attention + Residual Connection

        x = x + self.attention(self.ln1(x))

        # Feed Forward + Residual Connection

        x = x + self.ffn(self.ln2(x))

        return x

In [155]:
# Test the Transformer block

block = TransformerBlock(
    d_model = 128,
    num_heads = 4, 
    block_size = 64,
    ffn_dim=128
    )

output = block(x_emb)

print(output.shape)

torch.Size([4, 64, 128])


Our First Transformer Block is complete. Now we will build a multiple Transformer block model

In [160]:
# Multiple Transformer Block.

class TransformerStack(nn.Module):

    def __init__(self, d_model, num_heads, block_size, num_layers, ffn_dim):
        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                num_heads,
                block_size,
                ffn_dim
            )
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)

        return x

In [162]:
# Test The Stack

stack = TransformerStack(
    d_model = 128,
    num_heads = 4,
    block_size = 64,
    num_layers = 4,
    ffn_dim=512
)

output = stack(x_emb)

print(output.shape)

torch.Size([4, 64, 128])


### ACTUAL MINI-GPT Model

In [163]:
# Mini-GPT

class MiniGPT(nn.Module):

    def __init__(
            self,
            vocab_size,
            d_model,
            num_heads,
            num_layers,
            block_size,
            ffn_dim
    ):
        super().__init__()

        self.block_size = block_size

        # Token Embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Positional Embeddings
        self.position_embedding = nn.Embedding(
            block_size,
            d_model
        )

        # Transformer Blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                num_heads,
                block_size,
                ffn_dim
            )
            for _ in range(num_layers)
        ])

        # Final normalization
        self.ln_f = nn.LayerNorm(d_model)

        # Language Model head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, idx):

        B, T = idx.shape

        # Token Embeddings
        token_embeddings = self.token_embedding(idx)

        # Position Embeddings
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding(positions)

        # Combine Token and Position Information
        x = token_embeddings + position_embeddings

        # Transformer Blocks
        for block in self.blocks:
            x = block(x)

        # Final layerNorm
        x = self.ln_f(x)

        # Convert Representations to vocabulary logits
        logits = self.lm_head(x)

        return logits
        

In [164]:
print("MiniGPT Configuration")
print("----------------------")

for key, value in config.items():
    print(f"{key}: {value}")

MiniGPT Configuration
----------------------
vocab_size: 65
d_model: 128
num_heads: 8
num_layers: 4
block_size: 64
ffn_dim: 512


In [165]:
# Instantiate it:
model = MiniGPT(**config)

In [166]:
logits = model(x)

In [167]:
print(logits.shape)

torch.Size([32, 64, 65])


Now We connect the logits to the training target and we will calculate the loss

In [168]:
print(y.shape)

torch.Size([32, 64])


We will use cross entropy loss for calculating the loss here. It compares the probability assigned to the correct value. This loss heavily penalizes the error when the model is condidently wrong.

CrossEntropyLoss compares the model's logits against the correct target
class. It internally applies LogSoftmax and Negative Log-Likelihood,
so we do not need to apply Softmax before calculating the loss.

We will flatten the first two dimensions because the pytorch's `CrossEntropyLoss` expects the predictions in this general form [N, number_of_classes]

In [70]:
# We flatten the first two dimensions

B, T, C = logits.shape

logits_flat = logits.view(B * T, C)
targets_flat = y.view(B * T)

print(logits_flat.shape)
print(targets_flat.shape)

torch.Size([256, 65])
torch.Size([256])


In [71]:
loss_fn = nn.CrossEntropyLoss()

loss = loss_fn(
    logits_flat,
    targets_flat
)

print(loss.item())

4.412050724029541


The model is just in its initialization phase and is not yet trained. Getting this value is expected.

In [72]:
# We will make the model learn

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0003
)

In [73]:
# Our first Training step
optimizer.zero_grad()

loss.backward()

optimizer.step()

In [74]:
# Perform 1 forward pass

# Get a fresh batch
x, y = get_batch(
    train_ids,
    batch_size=4,
    block_size=64
)

In [75]:
logits = model(x)

B, T, C = logits.shape

loss = loss_fn(
    logits.view(B * T, C),
    y.view(B * T)
)

print("New loss:", loss.item())

New loss: 4.412290573120117


In [76]:
# Train the model for many steps

num_iterations = 100

for step in range(num_iterations):

    x, y = get_batch(
        train_ids,
        batch_size=32,
        block_size=64
    )

    logits = model(x)

    B, T, C = logits.shape

    loss = loss_fn(
        logits.view(B * T, C),
        y.view(B * T)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"Step {step}: Loss {loss.item(): .4f}")

Step 0: Loss  4.2693
Step 10: Loss  3.4400
Step 20: Loss  3.2802
Step 30: Loss  3.1052
Step 40: Loss  2.9979
Step 50: Loss  2.8213
Step 60: Loss  2.8423
Step 70: Loss  2.7050
Step 80: Loss  2.7445
Step 90: Loss  2.6824


This was our first miniature training step.

At this moment we are just understanding how the model is fitting the data. But we need to be cautious about overfitting. So we will have to create a proper evaluation function.

In [77]:
@torch.no_grad() # We are not learning so we dont need grads
def estimate_loss(model, train_ids, val_ids, eval_iters=20):
    model.eval()

    results = {}

    for split_name, data in [
        ("train", train_ids),
        ("val", val_ids)
    ]:

        losses = []

        for _ in range(eval_iters):

            x, y = get_batch(
                data,
                batch_size=32,
                block_size=64
            )

            logits = model(x)

            B, T, C = logits.shape

            loss = loss_fn(
                logits.view(B * T, C),
                y.view(B * T)
            )

            losses.append(loss.item())

        results[split_name] = sum(losses) / len(losses)

    model.train()

    return results

In [78]:
# Test the current model

losses = estimate_loss(
    model,
    train_ids,
    val_ids
)

In [79]:
print(losses)

{'train': 2.6285542488098144, 'val': 2.6229520201683045}


#### Proper Training Loop

In [131]:
# max_iters = 1000 This was before
# max_iters = 5000

max_iters = 10000
eval_interval = 100
batch_size = 32
block_size = 64

learning_rate = 3e-4

In [132]:
# Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

In [133]:
# Track the best validation loss

best_val_loss = float("inf")

In [134]:
# Proper Training loop

for step in range(max_iters):

    # Evaluate periodically
    if step % eval_interval == 0:
        losses = estimate_loss(
            model,
            train_ids,
            val_ids,
            eval_iters=20
        )

        print(
            f"Step {step}: "
            f"Train Loss {losses['train']:.4f}, "
            f"Val Loss {losses['val']:.4f}"
        )

        # Save best model
        if losses["val"] < best_val_loss:

            best_val_loss = losses["val"]

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "step": step,
                    "val_loss": best_val_loss
                },
                "best_model.pt"
            )

            print("  → Best model saved!")

    # Get training batch
    x, y = get_batch(
        train_ids,
        batch_size=batch_size,
        block_size=block_size
    )

    # Forward pass
    logits = model(x)

    B, T, C = logits.shape

    loss = loss_fn(
        logits.view(B * T, C),
        y.view(B * T)
    )

    # Backpropagation
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

Step 0: Train Loss 1.4974, Val Loss 1.6919
  → Best model saved!
Step 100: Train Loss 1.5156, Val Loss 1.6678
  → Best model saved!
Step 200: Train Loss 1.4950, Val Loss 1.6592
  → Best model saved!
Step 300: Train Loss 1.4836, Val Loss 1.6749
Step 400: Train Loss 1.4782, Val Loss 1.6733
Step 500: Train Loss 1.4643, Val Loss 1.6737
Step 600: Train Loss 1.4609, Val Loss 1.6723
Step 700: Train Loss 1.4794, Val Loss 1.6669
Step 800: Train Loss 1.4754, Val Loss 1.6540
  → Best model saved!
Step 900: Train Loss 1.4606, Val Loss 1.6414
  → Best model saved!
Step 1000: Train Loss 1.4659, Val Loss 1.6697
Step 1100: Train Loss 1.4581, Val Loss 1.6321
  → Best model saved!
Step 1200: Train Loss 1.4484, Val Loss 1.6696
Step 1300: Train Loss 1.4461, Val Loss 1.6443
Step 1400: Train Loss 1.4525, Val Loss 1.6449
Step 1500: Train Loss 1.4381, Val Loss 1.6519
Step 1600: Train Loss 1.4375, Val Loss 1.6421
Step 1700: Train Loss 1.4261, Val Loss 1.6260
  → Best model saved!
Step 1800: Train Loss 1.4279, 

#### Actual Text Generation Phase

In [84]:
# Defining the Generation Loop

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    model.eval()

    for _ in range(max_new_tokens):

        # Keep only the most recent block_size tokens
        idx_cond = idx[:, -model.block_size:]

        # Forward pass
        logits = model(idx_cond)

        # We only need predictions for the last position
        logits = logits[:, -1, :]

        # Temperature controls randomness
        logits = logits / temperature

        # Optional Top-K Sampling
        if top_k is not None:
            v, _ = torch.topk(
                logits,
                min(top_k, logits.size(-1))
            )
            logits[logits < v[:, [-1]]] = float("-inf")

        # Convert Logits into Probabilitires
        probs = F.softmax(logits, dim=-1)

        # Sample next character
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append Generated Token
        idx = torch.cat((idx, next_token), dim=1)

    model.train()

    return idx

In [135]:
# Load our best checkpoint

checkpoint = torch.load(
    "best_model.pt",
    map_location="cpu"
)

model.load_state_dict(checkpoint['model_state_dict'])

print("Loaded Checkpoint from step: ", checkpoint['step'])
print("Best Validation Loss: ", checkpoint["val_loss"])

Loaded Checkpoint from step:  9500
Best Validation Loss:  1.539136129617691


In [136]:
# Lets test it

prompt = "ROMEO:"
prompt_ids = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

attention = model.get_attention(
    prompt_ids,
    layer=0
)

generate_ids = generate(
    model,
    prompt_ids,
    max_new_tokens=500,
    temperature=0.8,
    top_k=20
)

generated_text = decode(generate_ids[0].tolist())

print(attention.shape)
print(generated_text)

torch.Size([1, 4, 6, 6])
ROMEO:
Go you say, and I hear my honesty.

MOMGONTES:
Say, I shall be here, I'll then oft me?

DUCHESSS OF YORK:
Are you, that there a disobey,
When I stay being thee, if I lighten
For sight of her his adventure, may be
That I have come deedfer'd ballad to myself,
Thou art that these sorrow by my head.

First Senator:
Go, my lord?

MENENIUS:
What do you were me assembland and the head.

DUCHESS OF YORK:
Good George the blood, to my untrold be but
A shall beat of the shall by the king,
Is a sin of a na


In [137]:
# Lets test it for different temperatures

prompt = "ROMEO:"
prompt_ids = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

for temperature in [0.3, 0.8, 1.0, 1.3]:
    generate_ids = generate(
        model,
        prompt_ids,
        max_new_tokens=500,
        temperature=temperature,
        top_k=20
    )

    generated_text = decode(generate_ids[0].tolist())

    print(f"\n{'='*20} Temperature: {temperature} {'='*20}\n")
    print(generated_text)


==================== Temperature: 0.3 ====================

ROMEO:
What is the like of all of his hands.

SICINIUS:
The hate of the king, and the world of his death,
To the stone of his and the stone of his country
To have a served and the earth.

DUKE VINCENTIO:
I the word of your heart and the day of the souls,
And the people of the state of your grace.

LADY CAPULET:
The common something of the wind of the world,
And the country of the cause of the people,
That she was far of the death of the way
To the was and off the great of my crown.

LUCIO:
I have a ma

==================== Temperature: 0.8 ====================

ROMEO:
Are your husband, were we dead, here was no eyes.

GLOUCESTER:
Talk, I beseech you at much are dewn, thy words than trust:
You have look trust me the king, at his alack,
That thinks, or be a wate
Unded to young marked sounds before England,
It not been his houses than your by roaring summers;
He will take the clouds with my hands,
If stooding some forth masters 

In [138]:
# Lets test it

prompt = "Jupyter"
prompt_ids = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

attention = model.get_attention(
    prompt_ids,
    layer=0
)

generate_ids = generate(
    model,
    prompt_ids,
    max_new_tokens=500,
    temperature=0.8,
    top_k=20
)

generated_text = decode(generate_ids[0].tolist())

print(attention.shape)
print(generated_text)

torch.Size([1, 4, 7, 7])
Jupyter, and I say, old the spide of
rebellion, and all go a forth, place between in sad my
hanging, and I should that is haste the norther eyes,
And we burn, if you see, would at a name;
That with my word deed to thee my life.

PAULINA:
Be stit:
Lascive of York!
I would crown faints in the one field:
The suspected when I fill your hands,
Both your father banish'd; and there's forget
To sufficers in his deed and gracious degrief,
That I am ground for the man-grown is great;
And, if thou art solemn at h


At this moment we trained the model for longer.

The Word generation is still better but the sentences makes no sense yet. 

In [139]:
# ============================================================
# EXPERIMENT: 8 Attention Heads
# ============================================================

num_heads = 8

model_8heads = MiniGPT(
    vocab_size=vocab_size,
    d_model=128,
    num_heads=num_heads,
    num_layers=4,
    block_size=64
)

loss_fn = nn.CrossEntropyLoss()

optimizer_8heads = torch.optim.AdamW(
    model_8heads.parameters(),
    lr=3e-4
)

max_iters = 10000
eval_interval = 100
batch_size = 32
block_size = 64

best_val_loss_8 = float("inf")
best_step_8 = None

for step in range(max_iters):

    if step % eval_interval == 0:

        losses = estimate_loss(
            model_8heads,
            train_ids,
            val_ids,
            eval_iters=20
        )

        print(
            f"Step {step}: "
            f"Train Loss {losses['train']:.4f}, "
            f"Val Loss {losses['val']:.4f}"
        )

        if losses["val"] < best_val_loss_8:

            best_val_loss_8 = losses["val"]
            best_step_8 = step

            torch.save(
                {
                    "model_state_dict": model_8heads.state_dict(),
                    "optimizer_state_dict": optimizer_8heads.state_dict(),
                    "step": step,
                    "val_loss": best_val_loss_8
                },
                "best_model_8heads.pt"
            )

            print("  → Best 8-head model saved!")

    x, y = get_batch(
        train_ids,
        batch_size=batch_size,
        block_size=block_size
    )

    logits = model_8heads(x)

    B, T, C = logits.shape

    loss = loss_fn(
        logits.view(B * T, C),
        y.view(B * T)
    )

    optimizer_8heads.zero_grad()

    loss.backward()

    optimizer_8heads.step()


print("\n==============================")
print("8-HEAD EXPERIMENT COMPLETE")
print("==============================")
print(f"Best Step: {best_step_8}")
print(f"Best Validation Loss: {best_val_loss_8:.4f}")

Step 0: Train Loss 4.3967, Val Loss 4.3871
  → Best 8-head model saved!
Step 100: Train Loss 2.6448, Val Loss 2.6605
  → Best 8-head model saved!
Step 200: Train Loss 2.5063, Val Loss 2.5256
  → Best 8-head model saved!
Step 300: Train Loss 2.4258, Val Loss 2.4412
  → Best 8-head model saved!
Step 400: Train Loss 2.3441, Val Loss 2.3521
  → Best 8-head model saved!
Step 500: Train Loss 2.2813, Val Loss 2.2958
  → Best 8-head model saved!
Step 600: Train Loss 2.2247, Val Loss 2.2450
  → Best 8-head model saved!
Step 700: Train Loss 2.1612, Val Loss 2.2049
  → Best 8-head model saved!
Step 800: Train Loss 2.1159, Val Loss 2.1729
  → Best 8-head model saved!
Step 900: Train Loss 2.0761, Val Loss 2.1085
  → Best 8-head model saved!
Step 1000: Train Loss 2.0238, Val Loss 2.0763
  → Best 8-head model saved!
Step 1100: Train Loss 1.9971, Val Loss 2.0627
  → Best 8-head model saved!
Step 1200: Train Loss 1.9486, Val Loss 2.0295
  → Best 8-head model saved!
Step 1300: Train Loss 1.9186, Val Los

In [140]:
prompt = "ROMEO:"

prompt_ids = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

generate_ids = generate(
    model_8heads,
    prompt_ids,
    max_new_tokens=500,
    temperature=0.8,
    top_k=20
)

generated_text = decode(generate_ids[0].tolist())

print(generated_text)

ROMEO:
I think your grace to be this confessed:
But what displace and starm'd in his mind;
Not do in this behold of a vice all like them.

First Was any hundress it?

NORFOLK:
In this?

MOPSA:
I carentain'd! for the proclaim of his king,
Meny own and nor to fresh is death, gentleman,
But the sorrow with their sistory and cheased
That well I know your son-tears and in there
At makes these mercy this houses to lose,
And let a book morrow like is a drop
To do in the issuit before his mine. Prove,
The per


We can observe that using an `8 head` transformer network perform way better and is able to learn more sensible words. But the meaning in the sentences is still missing. We can tweak the hyperparameters even more to get the desired result. But it is for later now.